In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
ddos_data=pd.read_csv('/kaggle/input/ddos-evaluation-dataset-cic-ddos2019/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')

# **Exploration des données**

In [ ]:
ddos_data.shape

In [ ]:
ddos_data.info()

**distibution de données**

In [ ]:
ddos_data[' Label'].value_counts()

In [ ]:
label_counts=ddos_data[' Label'].value_counts()
plt.figure(figsize=(3, 3))
plt.pie(label_counts, labels=label_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Distribution of DDoS Attack Types')
plt.axis('equal')
plt.show()

**convertir type de Timestamp**

In [ ]:
ddos_data[' Timestamp'] = pd.to_datetime(ddos_data[' Timestamp'])
ddos_data.sort_values(by=' Timestamp', inplace=True)

In [ ]:
ddos_data

**supprimer colonne inutile**

In [ ]:
constant_columns=ddos_data.columns[ddos_data.nunique()==1]
constant_columns
len(constant_columns)

In [ ]:
for column in constant_columns:
  print(f'la colonne {column} a une seule valeure {ddos_data[column].value_counts()}\n')

In [ ]:
ddos_data.drop(columns=constant_columns,inplace=True)

In [ ]:
ddos_data[[ 'Flow ID',' Destination IP',' Source IP', ' Destination Port',' Source Port',' Protocol']]

It seems like the 'Flow ID' column contains concatenated information about source IP, destination IP, destination port, and protocol. We can drop it since it won't add much value.

In [ ]:
ddos_data.drop('Flow ID',axis=1,inplace=True)

In [ ]:
ddos_data.info()

In [ ]:
numerical_features = [feature for feature in ddos_data.columns if ddos_data[feature].dtypes != 'O']
corr_matrix=ddos_data[numerical_features].corr()

In [ ]:
import seaborn as sns
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Heatmap of Numerical Features')
plt.show()

In [ ]:
perfect_corr = np.where(np.isclose(corr_matrix, 1.0, atol=1e-8) | np.isclose(corr_matrix, -1.0, atol=1e-8))
perfect_corr_pairs = [(corr_matrix.index[x], corr_matrix.columns[y]) for x, y in zip(*perfect_corr) if x != y]

# Suppression des doublons (i.e., (A, B) est identique à (B, A))
perfect_corr_pairs_set = set(tuple(sorted(pair)) for pair in perfect_corr_pairs)

# Convertir le set en liste pour l'affichage ou l'usage ultérieur
perfect_corr_pairs_list = list(perfect_corr_pairs_set)

# Afficher les paires de colonnes parfaitement corrélées
for pair in perfect_corr_pairs_list:
    print(pair)

(' ECE Flag Count', ' RST Flag Count'):Chaque élément représente un type différent de flag TCP.


(' Total Fwd Packets', 'Subflow Fwd Packets'):on peut supprimer 'Subflow Fwd Packets'


(' Fwd Header Length', ' Fwd Header Length.1'):redandance


(' Avg Fwd Segment Size', ' Fwd Packet Length Mean'):taille et langeur avg: supprimer une


(' Avg Bwd Segment Size', ' Bwd Packet Length Mean'): //


(' Subflow Bwd Bytes', ' Total Length of Bwd Packets'):nbre d'octet et la longeur : supprimer une


(' SYN Flag Count', 'Fwd PSH Flags'):Chaque élément représente un type différent de flag TCP.


(' Subflow Bwd Packets', ' Total Backward Packets'):on peut supprimer 'Subflow Bwd Packets'


(' Subflow Fwd Bytes', 'Total Length of Fwd Packets'):supprimer une


In [ ]:
ddos_data.drop('Subflow Fwd Packets',axis=1,inplace=True)
ddos_data.drop(' Fwd Header Length.1',axis=1,inplace=True)
ddos_data.drop(' Avg Fwd Segment Size',axis=1,inplace=True)
ddos_data.drop(' Avg Bwd Segment Size',axis=1,inplace=True)
ddos_data.drop(' Subflow Bwd Bytes',axis=1,inplace=True)
ddos_data.drop(' Subflow Bwd Packets',axis=1,inplace=True)
ddos_data.drop(' Subflow Fwd Bytes',axis=1,inplace=True)



# **Analyse des variables**

In [ ]:
connection_grouped_by_time=ddos_data.groupby([' Label',' Timestamp']).size().unstack(level=0)
plt.figure(figsize=(12, 6))
connection_grouped_by_time.plot()
plt.xlabel('Timestamp')
plt.ylabel('Number of Lines')
plt.xticks(rotation=90)
plt.title('Number of Lines per Timestamp')
plt.grid(axis='y')
plt.show()

In [ ]:
def viz_variable(variable):
    non_benign_data = ddos_data[ddos_data[' Label'] != 'BENIGN']
    non_benign_bytes_by_time = non_benign_data.groupby([' Timestamp', variable]).size().reset_index(name='count')
    non_benign_bytes_by_time = non_benign_bytes_by_time.groupby(' Timestamp')[variable].mean()

    benign_data = ddos_data[ddos_data[' Label'] == 'BENIGN']

    benign_bytes_by_time =benign_data.groupby([' Timestamp', variable]).size().reset_index(name='count')
    benign_bytes_by_time = benign_bytes_by_time.groupby(' Timestamp')[variable].mean()

    plt.figure(figsize=(12, 6))
    plt.plot(non_benign_bytes_by_time.index, non_benign_bytes_by_time.values, label='Non-Benign', linestyle='-', marker='o')
    plt.plot(benign_bytes_by_time.index, benign_bytes_by_time.values, label='Benign', linestyle='-', marker='o')
    plt.title('Average '+variable+ 'Received Over Time for non benign connections')
    plt.xlabel('Time')
    plt.ylabel('Average Bytes')
    plt.xticks(rotation=90)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:

binary_columns = []

for column in ddos_data.columns:
    if ddos_data[column].nunique() == 2:
            binary_columns.append(column)

binary_columns


In [ ]:
ddos_data.loc[ddos_data['Fwd PSH Flags'] != ddos_data[' PSH Flag Count'],['Fwd PSH Flags', ' PSH Flag Count']].size

In [ ]:
ddos_data.drop('Fwd PSH Flags',axis=1,inplace=True)


In [ ]:
numerical_features = [feature for feature in ddos_data.columns if (ddos_data[feature].dtypes != 'O' and feature not in binary_columns)]
numerical_features

In [ ]:
for i in numerical_features:
    if i !=' Timestamp':
        viz_variable(i)

on remarque pas de difference importante pour la variation des variables ' Flow IAT Mean' et' Flow IAT Min' dans le temps entre les connexions ddos et normale -> on les supprime

In [ ]:
ddos_data.drop(' Flow IAT Mean',axis=1,inplace=True)
ddos_data.drop(' Flow IAT Min',axis=1,inplace=True)

on remarque que ces variables :
 ' Fwd Packet Length Max',
 ' Fwd Packet Length Min',
 ' Fwd Packet Length Mean',
 ' Fwd Packet Length Std', donne la meme information  


In [ ]:
ddos_data.drop(' Fwd Packet Length Max',axis=1,inplace=True)
ddos_data.drop(' Fwd Packet Length Min',axis=1,inplace=True)
ddos_data.drop(' Fwd Packet Length Std',axis=1,inplace=True)

de meme pour:
 ' Bwd Packet Length Mean',
 ' Bwd Packet Length Std',

In [ ]:
ddos_data.drop(' Bwd Packet Length Std',axis=1,inplace=True)

et pour: 'Flow Bytes/s',
 ' Flow Packets/s',

In [ ]:
ddos_data.drop('Flow Bytes/s',axis=1,inplace=True)

 ' Flow IAT Std',
 ' Flow IAT Max',

In [ ]:
ddos_data.drop(' Flow IAT Std',axis=1,inplace=True)

 'Fwd IAT Total',
 ' Fwd IAT Mean',
 ' Fwd IAT Std',

In [ ]:
ddos_data.drop( ' Fwd IAT Std',axis=1,inplace=True)
ddos_data.drop(' Fwd IAT Mean',axis=1,inplace=True)

'Bwd IAT Total',
 ' Bwd IAT Mean',
 ' Bwd IAT Std',
 ' Bwd IAT Max',
 ' Bwd IAT Min',

In [ ]:
ddos_data.drop( ' Bwd IAT Std',axis=1,inplace=True)
ddos_data.drop(' Bwd IAT Max',axis=1,inplace=True)
ddos_data.drop( ' Bwd IAT Min',axis=1,inplace=True)


' Packet Length Std',
 ' Packet Length Variance',

In [ ]:
ddos_data.drop( ' Packet Length Variance',axis=1,inplace=True)


' Average Packet Size',  ' Packet Length Mean',

In [ ]:
ddos_data.drop(' Average Packet Size',axis=1,inplace=True)


'Active Mean',
 ' Active Std',
 ' Active Max',
 ' Active Min',

In [ ]:
ddos_data.drop(' Active Max',axis=1,inplace=True)
ddos_data.drop(' Active Min',axis=1,inplace=True)


'Idle Mean',
 ' Idle Std',
 ' Idle Max',
 ' Idle Min'

In [ ]:
ddos_data.drop(' Idle Max',axis=1,inplace=True)
ddos_data.drop(' Idle Min',axis=1,inplace=True)


In [ ]:
ddos_data.info()